# Day 9: The test set, once
This notebook runs the held-out test images (845 of them, never used for training, tuning or choosing a model) through your finished models.

**The rules, decided before you look at any result:**
1. **Declare your primary model first** (step 2). That is the model that goes in your demo and your resume bullet. It is chosen from validation results only.
2. **The test set is run once.** The notebook writes a lock file to Drive the first time it finishes and refuses to run again unless you deliberately override it. If you re-ran until a number looked better, it would no longer be a test.
3. **Report whatever comes out.** The numbers here will probably be a little lower than validation: the validation set was used to pick the best epoch and the best model, which flatters it slightly. A drop of a point or two is normal and not a problem. Do not retrain or retune because of these results; if you ever change something afterwards, say in the write-up that it was done after seeing the test set.

Use a GPU runtime (Runtime > Change runtime type > T4 GPU), though the test set is small enough to run on CPU too.

## 1. Setup

In [ ]:
import os, time, json, zipfile, datetime
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import torch, torchvision
from torch import nn
from torchvision import datasets, transforms
from PIL import Image
from scipy.stats import binomtest

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
from google.colab import drive
drive.mount("/content/drive")
CKPT_DIR = "/content/drive/MyDrive/maize_ckpt"
LOCK = f"{CKPT_DIR}/test_run_LOCK.json"

ROOT = "/content/maize_split"
if not os.path.isdir(ROOT + "/test"):
    z = "/content/drive/MyDrive/maize_split.zip"
    assert os.path.exists(z), "No /content/maize_split and no maize_split.zip on Drive: re-run Day 3 (and its optional Drive cell)."
    os.makedirs(ROOT, exist_ok=True)
    with zipfile.ZipFile(z) as zf:
        zf.extractall(ROOT)
print("ready")


## 2. Declare your choice, then check the lock
Set `PRIMARY` to the model you will present as *the* model. `EVAL` is the list of models to evaluate on the test set, each exactly once; the second one is there for the accuracy-versus-size comparison. The default list has the two Phase 2 models that were close on validation (`r18_layer4` is left out because `r18_layer3_4` is the same size and scored higher).

This cell stops with an error if `PRIMARY` is not set, or if the test set has already been run.

In [ ]:
PRIMARY = None                                  # <-- set this: "mnv2_last6" or "r18_layer3_4"
EVAL = ["mnv2_last6", "r18_layer3_4"]
FORCE = False                                   # only ever True if you accept this is NOT a clean test any more

assert PRIMARY in EVAL, 'Set PRIMARY above to one of the names in EVAL, e.g. PRIMARY = "mnv2_last6"'
if os.path.exists(LOCK) and not FORCE:
    prev = json.load(open(LOCK))
    raise RuntimeError(f"The test set was already evaluated on {prev['when']} (primary: {prev['primary']}). Use those results. "
                       "Re-running on the test set after seeing it would make the numbers unreliable.")
for name in EVAL:
    assert os.path.exists(f"{CKPT_DIR}/phase2_{name}_best.pt"), f"checkpoint for {name} not found in {CKPT_DIR}"
print("primary:", PRIMARY, "| evaluating:", EVAL)


## 3. Evaluate (one pass per model)

In [ ]:
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
eval_tf = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
test_ds = datasets.ImageFolder(ROOT + "/test", eval_tf)
classes = test_ds.classes
test_loader = torch.utils.data.DataLoader(test_ds, batch_size=128, shuffle=False, num_workers=2)
y = np.array(test_ds.targets)
print("test images:", len(test_ds), {c: int((y == i).sum()) for i, c in enumerate(classes)}, "(Day 3 summary: 248 / 313 / 284)")
H = classes.index("HEALTHY")

def build_model(arch):
    if arch == "resnet18":
        m = torchvision.models.resnet18(weights=None); m.fc = nn.Linear(m.fc.in_features, len(classes))
    else:
        m = torchvision.models.mobilenet_v2(weights=None); m.classifier[1] = nn.Linear(m.classifier[1].in_features, len(classes))
    return m

@torch.no_grad()
def predict(m):
    m.eval(); out = []
    for x, _ in test_loader:
        out.append(torch.softmax(m(x.to(device)), 1).cpu())
    return torch.cat(out).numpy()

def wilson(k, n, z=1.96):
    if n == 0: return 0.0, 1.0
    p = k / n; d = 1 + z * z / n; c = p + z * z / (2 * n); h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
    return (c - h) / d, (c + h) / d

def mcnemar(a, b, y):
    n1 = int(((a == y) & (b != y)).sum()); n2 = int(((a != y) & (b == y)).sum())
    return n1, n2, (binomtest(n1, n1 + n2, 0.5).pvalue if n1 + n2 else 1.0)

probs, R = {}, {}
for name in EVAL:
    ck = torch.load(f"{CKPT_DIR}/phase2_{name}_best.pt", map_location=device)
    assert ck["classes"] == classes
    m = build_model(ck["arch"]).to(device); m.load_state_dict(ck["model"])
    p = predict(m); probs[name] = p; pred = p.argmax(1)
    cm = np.zeros((3, 3), int); np.add.at(cm, (y, pred), 1)
    k = int((pred == y).sum()); lo, hi = wilson(k, len(y))
    rec = cm.diagonal() / cm.sum(1); prec = cm.diagonal() / np.maximum(cm.sum(0), 1)
    dis_t, dis_p = y != H, pred != H
    sens = (dis_t & dis_p).sum() / dis_t.sum(); spec = (~dis_t & ~dis_p).sum() / (~dis_t).sum()
    R[name] = dict(acc=k / len(y), lo=lo, hi=hi, cm=cm, rec=rec, prec=prec, sens=sens, spec=spec, val_acc=ck["val_acc"],
                   missed=int((dis_t & ~dis_p).sum()), false_alarm=int((~dis_t & dis_p).sum()))
    np.save(f"{CKPT_DIR}/test_probs_{name}.npy", p)
    print(f"\n=== {name} {'(PRIMARY)' if name == PRIMARY else ''} ===")
    print(f"test accuracy {k / len(y):.4f}  (95% range {lo:.3f} to {hi:.3f})   [validation accuracy was {ck['val_acc']:.4f}]")
    t = pd.DataFrame({"recall": rec, "precision": prec, "test images": cm.sum(1)}, index=classes).round(3)
    t["recall 95% range"] = [f"{wilson(int(cm[i, i]), int(cm[i].sum()))[0]:.3f} to {wilson(int(cm[i, i]), int(cm[i].sum()))[1]:.3f}" for i in range(3)]
    print(t)
    print("confusion matrix (rows true, columns predicted):")
    print(pd.DataFrame(cm, index=[f"true {c}" for c in classes], columns=[f"pred {c}" for c in classes]))
    print(f"healthy vs diseased: diseased leaves caught {sens:.3f} ({R[name]['missed']} of {int(dis_t.sum())} called healthy), "
          f"healthy leaves correctly cleared {spec:.3f} ({R[name]['false_alarm']} of {int((~dis_t).sum())} flagged as diseased)")
    del m

json.dump({"when": datetime.datetime.now().isoformat(timespec="seconds"), "primary": PRIMARY, "evaluated": EVAL,
           "test_acc": {n: R[n]["acc"] for n in EVAL}}, open(LOCK, "w"))
print("\nLock file written: the test set is now spent.")
if len(EVAL) == 2:
    a, b = EVAL
    n1, n2, pv = mcnemar(probs[a].argmax(1), probs[b].argmax(1), y)
    print(f"\nPaired comparison on the test images: {a} right & {b} wrong on {n1}; {b} right & {a} wrong on {n2}; McNemar p = {pv:.3f}")


## 4. Accuracy vs size vs speed
Pulled from the Phase 2 table on Drive. These are the numbers for the tradeoff talking point.

In [ ]:
p2 = pd.read_csv(f"{CKPT_DIR}/phase2_results.csv").set_index("name")
tab = pd.DataFrame({
    "test_acc": {n: R[n]["acc"] for n in EVAL}, "val_acc": {n: R[n]["val_acc"] for n in EVAL},
    "diseased_caught": {n: R[n]["sens"] for n in EVAL},
    "params_M": {n: p2.loc[n, "params_M"] for n in EVAL}, "size_MB": {n: p2.loc[n, "size_MB"] for n in EVAL},
    "cpu_ms_per_image": {n: p2.loc[n, "cpu_ms_per_image"] for n in EVAL}}).round(4)
print(tab)


## 5. Look at the mistakes (primary model)
The most confident mistakes come first, because those are the most informative. For each, look at the picture and write down what you see. Typical findings: a blurry or very close-up photo; a leaf with lesions from two causes at once; a very faint symptom; a photo where the label itself looks doubtful. Aim to note 15 to 20; they feed your error analysis and your limitations section.

In [ ]:
pp = probs[PRIMARY]; pred = pp.argmax(1); conf = pp.max(1)
wrong = np.where(pred != y)[0]
order = wrong[np.argsort(-conf[wrong])]
errs = pd.DataFrame({"file": [os.path.basename(test_ds.samples[i][0]) for i in order], "true": [classes[y[i]] for i in order],
                     "predicted": [classes[pred[i]] for i in order], "confidence": conf[order].round(3)})
errs.to_csv(f"{CKPT_DIR}/test_errors_{PRIMARY}.csv", index=False)
print(f"{len(wrong)} mistakes out of {len(y)} for {PRIMARY} (saved to Drive as test_errors_{PRIMARY}.csv)")
print(f"median confidence: correct {np.median(conf[pred == y]):.3f} vs mistakes {np.median(conf[wrong]):.3f}")

show = order[:20]
cols = 5; rows = int(np.ceil(len(show) / cols))
fig, ax = plt.subplots(rows, cols, figsize=(3.2 * cols, 3.6 * rows), squeeze=False)
for a in ax.ravel(): a.axis("off")
for k, i in enumerate(show):
    a = ax.ravel()[k]
    a.imshow(Image.open(test_ds.samples[i][0]).convert("RGB"))
    a.set_title(f"{os.path.basename(test_ds.samples[i][0])}\ntrue {classes[y[i]]} -> {classes[pred[i]]} ({conf[i]:.2f})", fontsize=8)
plt.tight_layout(); plt.show()
print(errs.head(20).to_string(index=False))


## 6. Save a summary
Paste `day9_summary.md` back into the chat, together with your notes on what the mistakes look like.

In [ ]:
lines = ["# Day 9 summary (test set, evaluated once)", f"- primary model (declared before testing): {PRIMARY}; evaluated: {', '.join(EVAL)}", f"- test images: {len(y)}", ""]
for n in EVAL:
    r = R[n]
    lines.append(f"- {n}: test accuracy {r['acc']:.4f} (95% range {r['lo']:.3f} to {r['hi']:.3f}); validation was {r['val_acc']:.4f}")
    lines.append("  - recall / precision: " + "; ".join(f"{c} {a:.3f} / {b:.3f}" for c, a, b in zip(classes, r["rec"], r["prec"])))
    lines.append(f"  - diseased caught {r['sens']:.3f} ({r['missed']} diseased called healthy); healthy cleared {r['spec']:.3f} ({r['false_alarm']} healthy flagged)")
    lines.append(f"  - confusion (rows true, columns predicted, order {', '.join(classes)}): {r['cm'].tolist()}")
    lines.append(f"  - size {tab.loc[n, 'size_MB']:.0f} MB, {tab.loc[n, 'params_M']:.1f}M params, {tab.loc[n, 'cpu_ms_per_image']:.0f} ms/image on CPU")
if len(EVAL) == 2:
    lines.append(f"- paired comparison of the two models on test: {n1} vs {n2} discordant images, McNemar p = {pv:.3f}")
lines.append(f"- mistakes by the primary model: {len(wrong)}; median confidence on mistakes {np.median(conf[wrong]):.3f} vs correct {np.median(conf[pred == y]):.3f}")
lines.append("- notes on the mistakes (fill in): ")
open("/content/day9_summary.md", "w").write("\n".join(lines) + "\n")
print("\n".join(lines))
try:
    from google.colab import files
    files.download("/content/day9_summary.md")
except ImportError:
    pass
